# Normalisation Pipeline — Raw Data to Model Output

We follow **one single node** from one case through every transformation,
in the exact order they happen from raw VTK file to final predicted temperature.

```
RAW DATA
   ↓  step 1 — build feature vector
RAW X  (x, y, z, q, k, T_fix)
   ↓  step 2 — log-transform q
X after log
   ↓  step 3 — z-score
X_norm  (what the network sees)
   ↓  step 4 — forward pass
T̂_norm  (network output)
   ↓  step 5 — denormalise
T_pred  [°C]  (final answer)
```

In [42]:
import sys, json
from pathlib import Path
import numpy as np
import torch
import meshio

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from arch import ThermalNet

ELMER_CASES = ROOT / 'elmer_cases' / 'thermal_ccx_beam'
SAVES       = ROOT / 'saves'
LOG_COLS    = [3]   # column index of q_total_mW

MATERIAL_K = {
    'Steel_A36':        50.0,
    'Steel_S355':       48.0,
    'Aluminium_6061':  167.0,
    'Titanium_Ti6Al4V':  6.7,
    'Concrete_C30':      1.8,
}

---
## Step 0 — Load the norm params

These were computed **once on the training split** during `train.py` and saved to disk.
They are never recomputed — inference always reloads the same file.

In [43]:
norm   = dict(np.load(SAVES / 'norm_params.npz'))
X_mean = norm['X_mean']   # (6,)
X_std  = norm['X_std']    # (6,)
Y_mean = norm['Y_mean']   # (1,)
Y_std  = norm['Y_std']    # (1,)

COLS = ['x [mm]', 'y [mm]', 'z [mm]', 'q [mW]', 'k [W/m/K]', 'T_fix [°C]']

print('Norm params (computed on training split only):')
print(f'  {"column":<14}  {"X_mean":>12}  {"X_std":>12}')
print('  ' + '-' * 42)
for i, col in enumerate(COLS):
    print(f'  {col:<14}  {X_mean[i]:12.4f}  {X_std[i]:12.4f}')
print(f'\n  Y_mean = {Y_mean[0]:.4f} °C')
print(f'  Y_std  = {Y_std[0]:.4f} °C')

Norm params (computed on training split only):
  column                X_mean         X_std
  ------------------------------------------
  x [mm]              499.1243      295.7636
  y [mm]               49.6726       34.0049
  z [mm]               49.6867       33.9719
  q [mW]                8.8269        1.5438
  k [W/m/K]            52.3443       58.8945
  T_fix [°C]           19.6026        0.3993

  Y_mean = 243.4540 °C
  Y_std  = 699.7877 °C


---
## Step 1 — Raw data: read one node from one case

We pick **case_0000** (Steel_A36, Q=500 mW, train split) and take **node 0** as our example.

The raw feature vector has 6 columns:

| col | feature | unit | source |
|-----|---------|------|--------|
| 0 | x | mm | VTK mesh coordinates |
| 1 | y | mm | VTK mesh coordinates |
| 2 | z | mm | VTK mesh coordinates |
| 3 | q_total | mW | case_params.json |
| 4 | k | mW/mm/°C | material lookup |
| 5 | T_fix | °C | case_params.json |

In [44]:
CASE_ID  = 'case_0000'
NODE_IDX = 0

case_dir = ELMER_CASES / CASE_ID
params   = json.loads((case_dir / 'case_params.json').read_text())
mesh     = meshio.read(str(case_dir / 'case.vtk'))

# raw coordinates of the chosen node
coord = mesh.points[NODE_IDX].astype(np.float32)   # (3,)

# case-level scalars (same for all nodes in this case)
q     = params['q_total_mW']      # mW
k     = MATERIAL_K[params['material']]  # mW/mm/°C  (= W/m/K numerically)
T_fix = params['T_fix_C']         # °C

# ground truth temperature at this node
temp_key = next(k2 for k2 in mesh.point_data if k2.lower() == 'temperature')
T_true   = float(mesh.point_data[temp_key][NODE_IDX])

# assemble raw feature vector
X_raw = np.array([coord[0], coord[1], coord[2], q, k, T_fix], dtype=np.float32)

print(f'Case    : {CASE_ID}  |  material: {params["material"]}  |  split: {params["split"]}')
print(f'Node    : {NODE_IDX}')
print()
print('RAW feature vector X_raw:')
for col, val in zip(COLS, X_raw):
    print(f'  {col:<14} = {val:>12.4f}')
print(f'\nGround truth T_true = {T_true:.4f} °C')

Case    : case_0000  |  material: Steel_A36  |  split: train
Node    : 0

RAW feature vector X_raw:
  x [mm]         =       0.0000
  y [mm]         =       0.0000
  z [mm]         =       0.0000
  q [mW]         =     500.0000
  k [W/m/K]      =      50.0000
  T_fix [°C]     =      20.0000

Ground truth T_true = 20.0000 °C


---
## Step 2 — Log-transform q (column 3)

**Why:** q spans 0.1 W to 500 W across the dataset — a 5000× range.
A z-score on raw q would be meaningless because the distribution is heavily skewed.
Taking log makes it approximately uniform/Gaussian before z-scoring.

Only column 3 (q) is log-transformed. All other columns are left as-is.

In [45]:
X_log = X_raw.copy()
X_log[3] = np.log(X_raw[3])   # log(q_total_mW)

print('After log-transform on column 3 (q):')
print(f'  q_raw      = {X_raw[3]:.4f} mW')
print(f'  log(q_raw) = {X_log[3]:.4f}')
print()
print('Full X_log vector:')
for col, raw, logged in zip(COLS, X_raw, X_log):
    changed = '  ← log applied' if col == 'q [mW]' else ''
    print(f'  {col:<14}  raw={raw:>12.4f}  after log={logged:>12.4f}{changed}')

After log-transform on column 3 (q):
  q_raw      = 500.0000 mW
  log(q_raw) = 6.2146

Full X_log vector:
  x [mm]          raw=      0.0000  after log=      0.0000
  y [mm]          raw=      0.0000  after log=      0.0000
  z [mm]          raw=      0.0000  after log=      0.0000
  q [mW]          raw=    500.0000  after log=      6.2146  ← log applied
  k [W/m/K]       raw=     50.0000  after log=     50.0000
  T_fix [°C]      raw=     20.0000  after log=     20.0000


---
## Step 3 — Z-score normalisation

```
X_norm = (X_log - X_mean) / X_std
```

X_mean and X_std were computed on the training split **after** the log-transform.
So the mean of column 3 is mean(log(q)), not mean(q).

After this step all columns should be approximately centred at 0 with std ≈ 1
— at least for training nodes. Val/test nodes may fall outside [-2, 2].

In [46]:
X_norm = (X_log - X_mean) / X_std

print('After z-score normalisation:')
print(f'  {"column":<14}  {"X_log":>10}  {"X_mean":>10}  {"X_std":>10}  {"X_norm":>10}')
print('  ' + '-' * 58)
for col, xl, xm, xs, xn in zip(COLS, X_log, X_mean, X_std, X_norm):
    print(f'  {col:<14}  {xl:10.4f}  {xm:10.4f}  {xs:10.4f}  {xn:10.4f}')

print()
print('X_norm is what the network receives as input.')
print('Values close to 0 = near the training mean.')
print('Values outside [-2, 2] = unusual / potentially extrapolation.')

After z-score normalisation:
  column               X_log      X_mean       X_std      X_norm
  ----------------------------------------------------------
  x [mm]              0.0000    499.1243    295.7636     -1.6876
  y [mm]              0.0000     49.6726     34.0049     -1.4607
  z [mm]              0.0000     49.6867     33.9719     -1.4626
  q [mW]              6.2146      8.8269      1.5438     -1.6921
  k [W/m/K]          50.0000     52.3443     58.8945     -0.0398
  T_fix [°C]         20.0000     19.6026      0.3993      0.9952

X_norm is what the network receives as input.
Values close to 0 = near the training mean.
Values outside [-2, 2] = unusual / potentially extrapolation.


### Quick check — what does a test case look like in normalised space?

Let's compare a training node vs an extrapolation test node (Q=500W)
to see how far outside the normalised range the test case falls.

---
## Step 3b — Y normalisation (temperature label)

X is not the only thing that gets normalised. The **output** Y (temperature in °C) is also z-scored before training:

```
Y_norm = (Y_raw - Y_mean) / Y_std
```

Y_mean and Y_std are computed on the training split temperatures — across all nodes of all training cases.
The network is trained to predict Y_norm, not Y in °C directly.

This matters because without it, the loss would be dominated by high-temperature cases
(e.g. Concrete at Q=100W can reach thousands of °C) and the network would ignore low-temperature cases.

In [47]:
# Y normalisation — applied to the label during training
T_norm_true = (T_true - Y_mean[0]) / Y_std[0]

print('Y normalisation:')
print(f'  T_true (raw)   = {T_true:.4f} °C')
print(f'  Y_mean         = {Y_mean[0]:.4f} °C   (mean temperature across ALL train nodes)')
print(f'  Y_std          = {Y_std[0]:.4f} °C   (std  temperature across ALL train nodes)')
print(f'  Y_norm         = ({T_true:.4f} - {Y_mean[0]:.4f}) / {Y_std[0]:.4f}')
print(f'                 = {T_norm_true:.6f}')
print()
print('This is what the network is trained to output for this node.')
print()
# show the range to make it intuitive
print('Intuition — what common temperatures look like in normalised space:')
print(f'  20°C  (T_fix, coldest)        → Y_norm = {(20   - Y_mean[0])/Y_std[0]:.3f}')
print(f'  {Y_mean[0]:.0f}°C (Y_mean, average)      → Y_norm = {0.0:.3f}')
print(f'  {Y_mean[0]+Y_std[0]:.0f}°C (Y_mean + Y_std)      → Y_norm = {1.0:.3f}')

Y normalisation:
  T_true (raw)   = 20.0000 °C
  Y_mean         = 243.4540 °C   (mean temperature across ALL train nodes)
  Y_std          = 699.7877 °C   (std  temperature across ALL train nodes)
  Y_norm         = (20.0000 - 243.4540) / 699.7877
                 = -0.319317

This is what the network is trained to output for this node.

Intuition — what common temperatures look like in normalised space:
  20°C  (T_fix, coldest)        → Y_norm = -0.319
  243°C (Y_mean, average)      → Y_norm = 0.000
  943°C (Y_mean + Y_std)      → Y_norm = 1.000


In [48]:
# test case: case_0104 = Steel_A36, Q=500 W = 500000 mW  (extrap-above)
params_test = json.loads((ELMER_CASES / 'case_0104' / 'case_params.json').read_text())
X_raw_test  = np.array([500.0, 50.0, 50.0,
                         params_test['q_total_mW'],
                         MATERIAL_K[params_test['material']],
                         params_test['T_fix_C']], dtype=np.float32)
X_log_test  = X_raw_test.copy()
X_log_test[3] = np.log(X_raw_test[3])
X_norm_test = (X_log_test - X_mean) / X_std

print('Comparison: training node vs extrapolation test node (same coords, different Q)')
print(f'  {"column":<14}  {"train X_norm":>14}  {"test X_norm":>14}  note')
print('  ' + '-' * 65)
for col, xt, xte in zip(COLS, X_norm, X_norm_test):
    note = '  ← WAY outside range' if abs(xte) > 3 else ''
    print(f'  {col:<14}  {xt:14.4f}  {xte:14.4f}{note}')

print(f'\nTrain Q = {X_raw[3]:.0f} mW   → X_norm[3] = {X_norm[3]:.3f}')
print(f'Test  Q = {X_raw_test[3]:.0f} mW  → X_norm[3] = {X_norm_test[3]:.3f}')

Comparison: training node vs extrapolation test node (same coords, different Q)
  column            train X_norm     test X_norm  note
  -----------------------------------------------------------------
  x [mm]                 -1.6876          0.0030
  y [mm]                 -1.4607          0.0096
  z [mm]                 -1.4626          0.0092
  q [mW]                 -1.6921          2.7825
  k [W/m/K]              -0.0398         -0.0398
  T_fix [°C]              0.9952          0.9952

Train Q = 500 mW   → X_norm[3] = -1.692
Test  Q = 500000 mW  → X_norm[3] = 2.782


---
## Step 4 — Forward pass through ThermalNet

The network receives X_norm (6 values) and outputs one value: **T̂_norm**,
the predicted temperature in normalised space.

The network has no knowledge of the normalisation — it just maps 6 numbers to 1 number.

In [49]:
# load trained model
ckpt  = torch.load(SAVES / 'thermal_pinn.pt', map_location='cpu', weights_only=True)
model = ThermalNet(hidden=ckpt['hidden'])
state = ckpt['model_state']
if any(k2.startswith('module.') for k2 in state):
    state = {k2[len('module.'):]: v for k2, v in state.items()}
model.load_state_dict(state)
model.eval()

# forward pass — single node
x_tensor  = torch.from_numpy(X_norm).unsqueeze(0)   # (1, 6)
with torch.no_grad():
    T_norm_pred = model(x_tensor).item()             # scalar

# normalised ground truth for comparison
T_norm_true = (T_true - Y_mean[0]) / Y_std[0]

print('Forward pass:')
print(f'  Input  X_norm  : {X_norm}')
print(f'  Output T̂_norm  : {T_norm_pred:.6f}   (normalised predicted temperature)')
print(f'  True   T̂_norm  : {T_norm_true:.6f}   (normalised ground truth)')
print(f'  Error in norm space: {abs(T_norm_pred - T_norm_true):.6f}')

Forward pass:
  Input  X_norm  : [-1.6875787  -1.4607482  -1.4625801  -1.692144   -0.03980567  0.99516875]
  Output T̂_norm  : -0.320422   (normalised predicted temperature)
  True   T̂_norm  : -0.319317   (normalised ground truth)
  Error in norm space: 0.001105


---
## Step 5 — Denormalise the output back to °C

The inverse of the Y z-score:
```
T_pred [°C] = T̂_norm × Y_std + Y_mean
```

This uses the **same** Y_mean and Y_std that were computed on training data and saved to `norm_params.npz`.

In [50]:
T_pred = T_norm_pred * Y_std[0] + Y_mean[0]

print('Denormalisation:')
print(f'  T̂_norm          = {T_norm_pred:.6f}')
print(f'  Y_std           = {Y_std[0]:.4f} °C')
print(f'  Y_mean          = {Y_mean[0]:.4f} °C')
print(f'  T̂_norm × Y_std  = {T_norm_pred * Y_std[0]:.4f} °C')
print(f'  + Y_mean        = {T_pred:.4f} °C   ← final prediction')
print()
print(f'  T_true          = {T_true:.4f} °C')
print(f'  |ΔT|            = {abs(T_pred - T_true):.4f} °C')

Denormalisation:
  T̂_norm          = -0.320422
  Y_std           = 699.7877 °C
  Y_mean          = 243.4540 °C
  T̂_norm × Y_std  = -224.2274 °C
  + Y_mean        = 19.2266 °C   ← final prediction

  T_true          = 20.0000 °C
  |ΔT|            = 0.7734 °C


---
## Full pipeline summary — one node at a glance

In [51]:
print('=' * 55)
print('FULL PIPELINE — one node')
print('=' * 55)
print(f'\nSTEP 1 — raw features')
for col, val in zip(COLS, X_raw):
    print(f'   {col:<14} = {val:.4f}')

print(f'\nSTEP 2 — log(q)  :  {X_raw[3]:.2f} mW  →  {X_log[3]:.4f}')

print(f'\nSTEP 3 — z-score :')
for col, xn in zip(COLS, X_norm):
    print(f'   {col:<14} → {xn:8.4f}')

print(f'\nSTEP 4 — network : X_norm → T̂_norm = {T_norm_pred:.6f}')

print(f'\nSTEP 5 — denorm  : {T_norm_pred:.6f} × {Y_std[0]:.2f} + {Y_mean[0]:.2f} = {T_pred:.4f} °C')

print(f'\nGround truth     : {T_true:.4f} °C')
print(f'Error            : {abs(T_pred - T_true):.4f} °C')
print('=' * 55)

FULL PIPELINE — one node

STEP 1 — raw features
   x [mm]         = 0.0000
   y [mm]         = 0.0000
   z [mm]         = 0.0000
   q [mW]         = 500.0000
   k [W/m/K]      = 50.0000
   T_fix [°C]     = 20.0000

STEP 2 — log(q)  :  500.00 mW  →  6.2146

STEP 3 — z-score :
   x [mm]         →  -1.6876
   y [mm]         →  -1.4607
   z [mm]         →  -1.4626
   q [mW]         →  -1.6921
   k [W/m/K]      →  -0.0398
   T_fix [°C]     →   0.9952

STEP 4 — network : X_norm → T̂_norm = -0.320422

STEP 5 — denorm  : -0.320422 × 699.79 + 243.45 = 19.2266 °C

Ground truth     : 20.0000 °C
Error            : 0.7734 °C


---
## Things worth noticing

### 1 — k is not log-transformed

k spans 1.8 to 167 W/m/K — almost 100× range, similar to q.
Yet only q gets log-transformed. Let's check the raw distributions.

In [52]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

k_values = list(MATERIAL_K.values())
q_values = np.logspace(np.log10(500), np.log10(100000), 100)   # training Q range in mW

fig, axes = plt.subplots(2, 2, figsize=(11, 6))
fig.suptitle('Raw vs log distributions for q and k', fontsize=12)

axes[0,0].hist(q_values, bins=20, color='steelblue')
axes[0,0].set_title('q raw [mW]'); axes[0,0].set_xlabel('mW')

axes[0,1].hist(np.log(q_values), bins=20, color='steelblue')
axes[0,1].set_title('log(q)  ← what the network sees'); axes[0,1].set_xlabel('log(mW)')

axes[1,0].bar(range(len(k_values)), k_values, color='darkorange')
axes[1,0].set_xticks(range(len(k_values)))
axes[1,0].set_xticklabels(list(MATERIAL_K.keys()), rotation=20, fontsize=8)
axes[1,0].set_title('k raw [W/m/K]  ← what the network sees')

axes[1,1].bar(range(len(k_values)), np.log(k_values), color='darkorange')
axes[1,1].set_xticks(range(len(k_values)))
axes[1,1].set_xticklabels(list(MATERIAL_K.keys()), rotation=20, fontsize=8)
axes[1,1].set_title('log(k)  ← not currently used')

plt.tight_layout()
out = SAVES / 'norm_q_k_distributions.png'
plt.savefig(out, dpi=120)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

print(f'\nk values: {k_values}')
print(f'k range: {min(k_values):.1f} – {max(k_values):.1f}  ({max(k_values)/min(k_values):.0f}× span)')
print(f'After z-score, k_norm for each material:')
for mat, kv in MATERIAL_K.items():
    kn = (kv - X_mean[4]) / X_std[4]
    print(f'  {mat:<22} k={kv:6.1f}  k_norm={kn:7.3f}')

Saved → saves/norm_q_k_distributions.png

k values: [50.0, 48.0, 167.0, 6.7, 1.8]
k range: 1.8 – 167.0  (93× span)
After z-score, k_norm for each material:
  Steel_A36              k=  50.0  k_norm= -0.040
  Steel_S355             k=  48.0  k_norm= -0.074
  Aluminium_6061         k= 167.0  k_norm=  1.947
  Titanium_Ti6Al4V       k=   6.7  k_norm= -0.775
  Concrete_C30           k=   1.8  k_norm= -0.858


/tmp/ipykernel_179256/661582946.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2 — T_fix is almost constant

T_fix = 20°C for every case in the dataset.
After z-scoring with X_std[5] ≈ 0.4, this column is always the same value.
It carries essentially no information.

In [53]:
print(f'T_fix raw value   : {T_fix} °C')
print(f'X_mean[5] (T_fix) : {X_mean[5]:.4f} °C')
print(f'X_std[5]  (T_fix) : {X_std[5]:.6f} °C   ← very small')
print(f'T_fix normalised  : {(T_fix - X_mean[5]) / X_std[5]:.4f}')
print()
print('Since T_fix is always 20°C, this column is always the same normalised value.')
print('The network could drop this input and nothing would change.')
print('It would only matter if you ever varied T_fix across cases.')

T_fix raw value   : 20.0 °C
X_mean[5] (T_fix) : 19.6026 °C
X_std[5]  (T_fix) : 0.399348 °C   ← very small
T_fix normalised  : 0.9952

Since T_fix is always 20°C, this column is always the same normalised value.
The network could drop this input and nothing would change.
It would only matter if you ever varied T_fix across cases.


### 4 — The physics loss lives entirely in normalised space

During training, the physics loss (`losses.py`) receives `X_norm` and calls `model(X_norm)` to get `T̂_norm`.
It then differentiates `T̂_norm` with respect to the **normalised coordinates** x̂, ŷ, ẑ — not the physical ones.

```
∂T̂_norm / ∂x̂    ←  derivative of normalised output w.r.t. normalised x
∂²T̂_norm / ∂x̂²  ←  second derivative, same space
```

This means the Laplacian residual is in **normalised units**, not °C/mm².
To convert back to physical units you need the chain rule:

```
∂²T / ∂x²  =  ∂²T̂_norm / ∂x̂²  ×  Y_std / X_std[0]²
```

During training this conversion is **not applied** — the physics loss penalises the normalised Laplacian directly.
The λ=10 weight therefore scales a normalised quantity, making it hard to interpret physically.

In [54]:
import torch

# load model if not already loaded
try:
    model
except NameError:
    ckpt  = torch.load(SAVES / 'thermal_pinn.pt', map_location='cpu', weights_only=True)
    model = ThermalNet(hidden=ckpt['hidden'])
    state = ckpt['model_state']
    if any(k2.startswith('module.') for k2 in state):
        state = {k2[len('module.'):]: v for k2, v in state.items()}
    model.load_state_dict(state)
    model.eval()

# ── compute the Laplacian at our single node, step by step ───────────────────
x_tensor = torch.from_numpy(X_norm).unsqueeze(0).requires_grad_(True)  # (1, 6)

with torch.enable_grad():
    T_hat_norm = model(x_tensor)   # (1, 1)  — normalised output

    # first spatial derivatives in normalised space
    g = torch.autograd.grad(T_hat_norm.sum(), x_tensor, create_graph=True)[0]  # (1, 6)
    dT_dx_hat = g[0, 0].item()
    dT_dy_hat = g[0, 1].item()
    dT_dz_hat = g[0, 2].item()

    # second derivatives (diagonal Hessian) in normalised space
    lap_norm = 0.0
    for j in range(3):
        g2 = torch.autograd.grad(g[0, j], x_tensor,
                                  retain_graph=(j < 2),
                                  create_graph=False)[0]
        lap_norm += g2[0, j].item()

# convert to physical units using chain rule
sx = X_std[:3]                      # std of x, y, z  [mm]
lap_physical = sum(
    torch.autograd.grad(
        torch.autograd.grad(
            model(x_tensor.detach().clone().requires_grad_(True)).sum(),
            x_tensor.detach().clone().requires_grad_(True),
            create_graph=True
        )[0][0, j],
        x_tensor.detach().clone().requires_grad_(True),
        create_graph=False
    )[0][0, j].item() / (sx[j] ** 2)
    for j in range(3)
) if False else lap_norm / (sx ** 2).sum()   # simplified approximation for display

print('Physics loss normalisation — at our single node:')
print()
print('  STEP A — forward pass in normalised space:')
print(f'    X_norm fed to network → T̂_norm = {T_hat_norm.item():.6f}')
print()
print('  STEP B — first derivatives in normalised space (∂T̂_norm / ∂x̂):')
print(f'    ∂T̂/∂x̂ = {dT_dx_hat:.6f}')
print(f'    ∂T̂/∂ŷ = {dT_dy_hat:.6f}')
print(f'    ∂T̂/∂ẑ = {dT_dz_hat:.6f}')
print()
print('  STEP C — Laplacian in normalised space:')
print(f'    ∇²T̂_norm = {lap_norm:.6e}   (this is what the training loss penalises)')
print()
print('  STEP D — convert to physical units (chain rule: divide by X_std[j]²):')
print(f'    X_std[:3] = {sx}  mm')
for j, (name, sxj) in enumerate(zip(["x","y","z"], sx)):
    print(f'    X_std[{j}]² = {sxj**2:.2f} mm²')
print(f'    ∇²T in physical units ≈ ∇²T̂_norm / mean(X_std²) = {lap_norm / (sx**2).mean():.4e} °C/mm²')
print()
print('  NOTE: during training, step D is NOT done.')
print('  The loss penalises the normalised Laplacian directly.')
print('  λ=10 is weighting a normalised, unit-less quantity.')

Physics loss normalisation — at our single node:

  STEP A — forward pass in normalised space:
    X_norm fed to network → T̂_norm = -0.320422

  STEP B — first derivatives in normalised space (∂T̂_norm / ∂x̂):
    ∂T̂/∂x̂ = 0.002042
    ∂T̂/∂ŷ = -0.000025
    ∂T̂/∂ẑ = 0.000113

  STEP C — Laplacian in normalised space:
    ∇²T̂_norm = -7.883843e-04   (this is what the training loss penalises)

  STEP D — convert to physical units (chain rule: divide by X_std[j]²):
    X_std[:3] = [295.76358  34.0049   33.97192]  mm
    X_std[0]² = 87476.09 mm²
    X_std[1]² = 1156.33 mm²
    X_std[2]² = 1154.09 mm²
    ∇²T in physical units ≈ ∇²T̂_norm / mean(X_std²) = -2.6342e-08 °C/mm²

  NOTE: during training, step D is NOT done.
  The loss penalises the normalised Laplacian directly.
  λ=10 is weighting a normalised, unit-less quantity.


### 3 — Training vs val vs test in normalised space

The norm params are computed on training data only.
What does a test extrapolation case look like to the network?

In [55]:
examples = [
    ('case_0000', 'Steel_A36  Q=0.5W   train'),
    ('case_0102', 'Steel_A36  Q=52.5W  test-interp'),
    ('case_0103', 'Steel_A36  Q=200W   test-extrap-above'),
    ('case_0104', 'Steel_A36  Q=500W   test-extrap-above'),
    ('case_0310', 'Aluminium  Q=0.1W   test-extrap-below'),
    ('case_0314', 'Aluminium  Q=500W   test-extrap-above'),
]

print(f'  {"case / description":<42}  {"q_norm":>8}  {"k_norm":>8}')
print('  ' + '-' * 64)
for cid, desc in examples:
    p   = json.loads((ELMER_CASES / cid / 'case_params.json').read_text())
    kv  = MATERIAL_K[p['material']]
    q_n = (np.log(p['q_total_mW']) - X_mean[3]) / X_std[3]
    k_n = (kv - X_mean[4]) / X_std[4]
    flag = '  ← outside [-2, 2]' if abs(q_n) > 2 else ''
    print(f'  {desc:<42}  {q_n:8.3f}  {k_n:8.3f}{flag}')

  case / description                            q_norm    k_norm
  ----------------------------------------------------------------
  Steel_A36  Q=0.5W   train                     -1.692    -0.040
  Steel_A36  Q=52.5W  test-interp                1.323    -0.040
  Steel_A36  Q=200W   test-extrap-above          2.189    -0.040  ← outside [-2, 2]
  Steel_A36  Q=500W   test-extrap-above          2.782    -0.040  ← outside [-2, 2]
  Aluminium  Q=0.1W   test-extrap-below         -2.735     1.947  ← outside [-2, 2]
  Aluminium  Q=500W   test-extrap-above          2.782     1.947  ← outside [-2, 2]


---
## Recap — the full normalisation chain

```
RAW X                →  [x, y, z, q_mW, k, T_fix]
   ↓ log col 3
X_log                →  [x, y, z, log(q), k, T_fix]
   ↓ z-score  (train-only mean/std)
X_norm               →  what the network receives as input

RAW Y                →  T [°C]
   ↓ z-score  (train-only Y_mean, Y_std)
Y_norm               →  what the network is trained to predict

   ↓ network forward pass
T̂_norm  (output in normalised space)

   ↓ denormalise
T_pred [°C]          →  T̂_norm × Y_std + Y_mean

── physics loss (training only) ──────────────────────────
   X_norm → model → T̂_norm
      ↓ autograd ∂/∂x̂, ∂/∂ŷ, ∂/∂ẑ   (normalised coords)
   ∇²T̂_norm   (normalised Laplacian — what λ=10 penalises)
      ↓ ÷ X_std[j]²  (chain rule, only in notebook/eval)
   ∇²T  [°C/mm²]   (physical units — NOT used during training)
```

**Four things to keep in mind:**

| Observation | Impact |
|-------------|--------|
| k is not log-transformed despite 100× range | Aluminium/Concrete are compressed in feature space vs Steel |
| T_fix is constant → near-zero std | Column always normalises to the same value — effectively unused |
| Val MSE during training is in normalised space | Not directly comparable to MAE [°C] from inference |
| Physics loss penalises normalised Laplacian | λ=10 weights a unit-less quantity — hard to interpret physically |